# Data processing

Load a versioned CLI export and normalize it for exploration. This notebook never connects to Reddit or Gemini and does not write unless `PROCESSED_PATH` is set.

In [ ]:
MINERAL = "gold"
EXPORT_PATH = "exports/gold.jsonl"
PROCESSED_PATH = None  # Example: "exports/gold_processed.csv"

In [ ]:
import json
from pathlib import Path

import pandas as pd

working_directory = Path.cwd().resolve()
project_root = (
    working_directory.parent if working_directory.name == "notebooks" else working_directory
)
export_path = Path(EXPORT_PATH)
if not export_path.is_absolute():
    export_path = project_root / export_path
export_path

In [ ]:
records = []
if not export_path.exists():
    print(
        f"No export found at {export_path}. Create one with: "
        f"reddit-minerals export --mineral {MINERAL} --format jsonl "
        f"--output {EXPORT_PATH}"
    )
elif export_path.suffix.casefold() == ".jsonl":
    with export_path.open(encoding="utf-8") as export_file:
        records = [json.loads(line) for line in export_file if line.strip()]
else:
    with export_path.open(encoding="utf-8") as export_file:
        document = json.load(export_file)
    records = document.get("records", []) if isinstance(document, dict) else document

raw = pd.DataFrame.from_records(records)
print(f"Loaded {len(raw):,} records")

In [ ]:
processed = pd.json_normalize(records, sep="_") if records else raw.copy()

if "mineral" in processed.columns:
    processed = processed.loc[
        processed["mineral"].astype("string").str.casefold() == MINERAL.casefold()
    ].copy()

for timestamp_column in ("created_at", "updated_at", "analysed_at", "analyzed_at"):
    if timestamp_column in processed.columns:
        processed[timestamp_column] = pd.to_datetime(
            processed[timestamp_column], errors="coerce", utc=True
        )

processed = processed.drop_duplicates().reset_index(drop=True)
processed.head()

In [ ]:
if PROCESSED_PATH and not processed.empty:
    processed_path = Path(PROCESSED_PATH)
    if not processed_path.is_absolute():
        processed_path = project_root / processed_path
    processed_path.parent.mkdir(parents=True, exist_ok=True)
    processed.to_csv(processed_path, index=False)
    print(f"Wrote {len(processed):,} rows to {processed_path}")
else:
    print("No file written. Set PROCESSED_PATH to opt in.")